# Step 1: Data Models with Pydantic

**Goal:** Define the shape of data flowing in and out of the system — before writing a single API call.

---

## Why define models first?

When you're calling 3 different APIs (OpenAI, Gemini, Claude), each returns data in a slightly different structure.
By defining a **common response shape upfront**, your rest of the code only ever deals with *your* shape — not theirs.
This is called **normalizing** the API responses.

---

## Why Pydantic?

Python is dynamically typed — you can pass anything anywhere. That's great for flexibility, but bad when you're building
something with multiple moving parts (3 APIs, async calls, analytics). Pydantic gives you:

| Feature | What it does |
|---|---|
| **Validation** | Raises clear errors if wrong types are passed (e.g. `max_tokens="hello"`) |
| **Defaults** | Fields with defaults don't need to be passed explicitly |
| **Serialization** | `.model_dump()` → dict, `.model_dump_json()` → JSON string |
| **IDE autocomplete** | Because types are declared, your editor knows what fields exist |

### When to use Pydantic vs plain dataclasses vs dicts:

| Situation | Use |
|---|---|
| Data crosses API or system boundaries (input validation, parsing) | **Pydantic** |
| Simple internal data containers, no validation needed | `dataclass` |
| Throwaway / one-off data | `dict` |

**Rule of thumb:** If the data comes from *outside* your code (user input, API response, env vars), use Pydantic.

---

## The models we created

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field

class Provider(str, Enum):
    OPENAI = "openai"
    GEMINI = "gemini"
    CLAUDE = "claude"

### Why `str, Enum` instead of just `Enum`?

```python
# Just Enum:
print(Provider.OPENAI)        # → Provider.OPENAI
print(Provider.OPENAI.value)  # → openai

# str + Enum:
print(Provider.OPENAI)        # → openai   ← cleaner for printing / JSON
Provider.OPENAI == "openai"   # → True     ← can compare directly to strings
```

Since we'll display provider names in the UI and serialize them to JSON, `str, Enum` is cleaner.

In [ ]:
class ChatMessage(BaseModel):
    role: str      # "user" or "assistant"
    content: str


# Example: creating a message
msg = ChatMessage(role="user", content="What is the capital of France?")
print(msg)
print(msg.model_dump())  # → dict

In [ ]:
class ChatRequest(BaseModel):
    messages: list[ChatMessage]
    max_tokens: int = Field(default=1024, gt=0, le=8192)


# Field() lets you add constraints beyond just the type:
#   gt=0    → must be greater than 0
#   le=8192 → must be less than or equal to 8192
#
# Try passing an invalid value — Pydantic will raise a ValidationError:
# ChatRequest(messages=[msg], max_tokens=-5)  # ← will fail loudly

In [ ]:
class ChatResponse(BaseModel):
    provider: Provider
    model: str
    content: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    latency_seconds: float = 0.0
    estimated_cost_usd: float = 0.0
    error: str | None = None   # None = success; a string = something failed

    @property
    def total_tokens(self) -> int:
        return self.prompt_tokens + self.completion_tokens

### Key design decisions in `ChatResponse`:

**1. `error: str | None = None`**

We're calling 3 APIs in parallel. If Claude fails (e.g. bad API key), we don't want the whole program to crash.
Instead, Claude's response just has `error="Invalid API key"` while OpenAI and Gemini succeed normally.

**2. `@property` for `total_tokens`**

It's always `prompt_tokens + completion_tokens`, so there's no need to store it separately.
A `@property` computes it on the fly when you access `.total_tokens`.

**3. All numeric fields default to 0**

When constructing a response, you might not have token counts yet (e.g. if there's an error). Defaults prevent
needing to pass every field every time.

In [ ]:
# Simulating a successful response
success = ChatResponse(
    provider=Provider.OPENAI,
    model="gpt-4o-mini",
    content="Paris",
    prompt_tokens=15,
    completion_tokens=3,
    latency_seconds=0.842,
    estimated_cost_usd=0.000003,
)
print(success)
print("Total tokens:", success.total_tokens)

# Simulating a failed response
failure = ChatResponse(
    provider=Provider.CLAUDE,
    model="claude-haiku-4-5-20251001",
    content="",
    error="Authentication error: Invalid API key",
)
print(failure)

---

## Summary

- **`Provider`** — enum for the 3 providers; `str, Enum` so it serializes cleanly
- **`ChatMessage`** — one turn in a conversation (`role` + `content`)
- **`ChatRequest`** — what we send to any API (list of messages + max tokens)
- **`ChatResponse`** — normalized response from any API; has `error` field so failures don't crash the run

**Next step:** `core/config.py` — loading API keys from `.env` and defining model names + pricing rates.